# Evaluasi Metode Content-Based Filtering (CBF)

Notebook ini melakukan pengujian performa algoritma CBF menggunakan metrik:
- **RMSE** (Root Mean Square Error)
- **MAE** (Mean Absolute Error)

**Metodologi:**
Data rating dibagi menjadi train set dan test set (80:20).
Sistem CBF akan memprediksi skor relevansi film pada test set berdasarkan kemiripan genre (cosine similarity) dengan film yang pernah dirating tinggi oleh pengguna pada train set.
Skor prediksi kemudian dibandingkan dengan rating asli pada test set untuk menghitung RMSE dan MAE.

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

print('Library berhasil diimport.')

Library berhasil diimport.


## 2. Load Dataset

In [2]:
movies = pd.read_csv('data/movies.csv')
ratings = pd.read_csv('data/ratings.csv')

# Bersihkan film tanpa genre
movies_clean = movies[movies['genres'] != '(no genres listed)'].copy()

print(f'Jumlah film (setelah dibersihkan): {len(movies_clean)}')
print(f'Jumlah total rating              : {len(ratings)}')
print(f'Jumlah pengguna unik             : {ratings["userId"].nunique()}')

movies_clean.head(3)

Jumlah film (setelah dibersihkan): 9708
Jumlah total rating              : 100836
Jumlah pengguna unik             : 610


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


## 3. Ekstraksi Fitur Genre (One-Hot Encoding)

In [3]:
movies_clean['genres_list'] = movies_clean['genres'].str.split('|')

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies_clean['genres_list'])

# Hitung cosine similarity antar seluruh film
cosine_sim = cosine_similarity(genre_matrix)

# Buat indeks dari movieId ke posisi baris dalam matrix
movie_id_to_idx = pd.Series(range(len(movies_clean)), index=movies_clean['movieId'])

print(f'Ukuran matriks genre     : {genre_matrix.shape}')
print(f'Ukuran matriks similarity: {cosine_sim.shape}')

Ukuran matriks genre     : (9708, 19)
Ukuran matriks similarity: (9708, 9708)


## 4. Fungsi Prediksi CBF

Fungsi ini memprediksi rating sebuah film target untuk seorang pengguna.
Prediksi dihitung sebagai rata-rata tertimbang dari rating pengguna terhadap film-film yang sudah ditonton, dengan bobot berupa nilai cosine similarity antara film target dan film yang sudah ditonton.

In [4]:
def predict_cbf_rating(user_train_ratings, target_movie_id, movie_id_to_idx, cosine_sim, rating_scale=(0.5, 5.0)):
    """
    Memprediksi rating untuk target_movie_id berdasarkan riwayat rating
    pengguna pada train set menggunakan weighted average cosine similarity.
    """
    if target_movie_id not in movie_id_to_idx.index:
        return None
    
    target_idx = movie_id_to_idx[target_movie_id]
    
    numerator   = 0.0
    denominator = 0.0
    
    for _, row in user_train_ratings.iterrows():
        source_movie_id = row['movieId']
        actual_rating   = row['rating']
        
        if source_movie_id not in movie_id_to_idx.index:
            continue
        
        source_idx  = movie_id_to_idx[source_movie_id]
        sim_score   = cosine_sim[target_idx][source_idx]
        
        numerator   += sim_score * actual_rating
        denominator += abs(sim_score)
    
    if denominator == 0:
        return None
    
    predicted = numerator / denominator
    # Klip prediksi agar tetap berada dalam skala rating
    predicted = np.clip(predicted, rating_scale[0], rating_scale[1])
    return predicted

print('Fungsi prediksi CBF siap digunakan.')

Fungsi prediksi CBF siap digunakan.


## 5. Train-Test Split (80:20)

In [5]:
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

print(f'Ukuran Train Set : {len(train_data)} rating')
print(f'Ukuran Test Set  : {len(test_data)} rating')

Ukuran Train Set : 80668 rating
Ukuran Test Set  : 20168 rating


## 6. Proses Evaluasi (Menghitung Prediksi per Pengguna)

In [6]:
actuals    = []
predicted_ratings = []
result_rows = []

test_users = test_data['userId'].unique()
total      = len(test_users)

print(f'Memulai evaluasi untuk {total} pengguna pada test set...')

for i, user_id in enumerate(test_users):
    if (i + 1) % 50 == 0:
        print(f'  Memproses pengguna ke-{i+1} dari {total}...')

    user_train = train_data[train_data['userId'] == user_id]
    user_test  = test_data[test_data['userId'] == user_id]

    if user_train.empty:
        continue

    for _, test_row in user_test.iterrows():
        target_movie_id = test_row['movieId']
        actual_rating   = test_row['rating']

        predicted = predict_cbf_rating(
            user_train_ratings = user_train,
            target_movie_id    = target_movie_id,
            movie_id_to_idx    = movie_id_to_idx,
            cosine_sim         = cosine_sim
        )

        if predicted is not None:
            actuals.append(actual_rating)
            predicted_ratings.append(predicted)
            result_rows.append({
                'userId'          : user_id,
                'movieId'         : target_movie_id,
                'actual_rating'   : actual_rating,
                'predicted_rating': round(predicted, 4),
                'error'           : round(abs(actual_rating - predicted), 4)
            })

print(f'\nEvaluasi selesai. Total prediksi yang berhasil: {len(actuals)}')

Memulai evaluasi untuk 610 pengguna pada test set...


  Memproses pengguna ke-50 dari 610...


  Memproses pengguna ke-100 dari 610...


  Memproses pengguna ke-150 dari 610...


  Memproses pengguna ke-200 dari 610...


...

  Memproses pengguna ke-400 dari 610...


  Memproses pengguna ke-450 dari 610...


  Memproses pengguna ke-500 dari 610...


  Memproses pengguna ke-550 dari 610...


  Memproses pengguna ke-600 dari 610...

Evaluasi selesai. Total prediksi yang berhasil: 20137


## 7. Menghitung RMSE dan MAE

In [7]:
rmse = np.sqrt(mean_squared_error(actuals, predicted_ratings))
mae  = mean_absolute_error(actuals, predicted_ratings)

print('=' * 40)
print('  HASIL EVALUASI CBF (80:20 SPLIT)')
print('=' * 40)
print(f'  Jumlah data prediksi : {len(actuals)}')
print(f'  RMSE                 : {rmse:.4f}')
print(f'  MAE                  : {mae:.4f}')
print('=' * 40)

  HASIL EVALUASI CBF (80:20 SPLIT)
  Jumlah data prediksi : 20137
  RMSE                 : 0.9241
  MAE                  : 0.7159


## 8. Ekspor Hasil ke Excel

In [8]:
# DataFrame hasil prediksi detail
df_results = pd.DataFrame(result_rows)

# DataFrame ringkasan metrik
df_summary = pd.DataFrame({
    'Metrik'              : ['RMSE', 'MAE'],
    'Nilai'               : [round(rmse, 4), round(mae, 4)],
    'Keterangan'          : [
        'Root Mean Square Error (lebih sensitif terhadap error besar)',
        'Mean Absolute Error (rata-rata selisih prediksi vs aktual)'
    ]
})

output_file = 'hasil_evaluasi_cbf.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Ringkasan Metrik', index=False)
    df_results.to_excel(writer, sheet_name='Detail Prediksi',  index=False)

print(f'File berhasil disimpan: {output_file}')
print(f'  - Sheet 1: Ringkasan Metrik (RMSE & MAE)')
print(f'  - Sheet 2: Detail Prediksi ({len(df_results)} baris data)')

File berhasil disimpan: hasil_evaluasi_cbf.xlsx
  - Sheet 1: Ringkasan Metrik (RMSE & MAE)
  - Sheet 2: Detail Prediksi (20137 baris data)


## 9. Tampilan Awal Hasil

In [9]:
print('Ringkasan Metrik:')
display(df_summary)

print('\n10 Baris Pertama Detail Prediksi:')
display(df_results.head(10))

Ringkasan Metrik:


,Metrik,Nilai,Keterangan
0,RMSE,0.9241,Root Mean Square Error (lebih sensitif terhada...
1,MAE,0.7159,Mean Absolute Error (rata-rata selisih prediks...



10 Baris Pertama Detail Prediksi:


,userId,movieId,actual_rating,predicted_rating,error
0,432,77866.0,4.5,3.6775,0.8225
1,432,7306.0,3.0,3.4708,0.4708
2,432,81417.0,4.0,3.8902,0.1098
3,432,48516.0,2.5,3.6963,1.1963
4,432,72641.0,3.5,3.7881,0.2881
5,432,64839.0,4.0,3.7881,0.2119
6,432,1036.0,2.0,3.5761,1.5761
7,432,1801.0,4.5,3.6514,0.8486
8,432,589.0,3.5,3.5319,0.0319
9,432,3082.0,3.0,3.5397,0.5397
